# 마음결 AI — Qwen3-4B 실제 GPU 추가 학습
이 노트북은 **실제 CUDA GPU를 사용해 Qwen3-4B-Instruct-2507에 상담 분야 QLoRA 미세조정**을 실행합니다. 학습이 끝나면 `adapter_model.safetensors`와 학습 메타데이터를 저장합니다.

**중요:** 지금 포함된 예시는 인공적으로 작성한 작은 시험용 데이터입니다. GPU에서 학습을 마쳐도 ‘석사급 전문가’나 실제 정신건강 전문가와 동등하다는 의미가 아닙니다. 사람의 전문적 평가와 안전성 검증이 필요합니다.

1. Colab 상단 **런타임 → 런타임 유형 변경 → T4 GPU/L4/A100**을 선택합니다.
2. 메뉴의 **런타임 → 모두 실행**을 클릭합니다.
3. 완료 후 모델 어댑터 ZIP 파일을 저장합니다. 유료 GPU 서버가 없다면 인터넷에 상시 제공할 수는 없습니다.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU가 없습니다. 런타임 → 런타임 유형 변경 → GPU로 전환하세요.'
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

In [ ]:
import os
%cd /content
!git clone --depth 1 https://github.com/minjunmon109/maumgyeol-ai.git
%cd /content/maumgyeol-ai
!python -m pip install -q -r training/requirements-gpu.txt

In [ ]:
!python training/build_synthetic.py
!python training/train_qlora.py --epochs 2 --max-length 768

In [ ]:
from pathlib import Path
import json
out=Path('/content/maumgyeol-ai/training/outputs/maumgyeol-qwen3-lora')
assert (out/'adapter_model.safetensors').exists(), '학습이 성공하지 않았습니다.'
assert (out/'adapter_config.json').exists(), 'LoRA 설정 파일이 없습니다.'
manifest=json.loads((out/'training_manifest.json').read_text())
assert manifest['weight_file_present'] and manifest['steps']>0
print('학습 기록:')
print(json.dumps(manifest, ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
import os, zipfile
path='/content/maumgyeol-trained-lora.zip'
with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for root, _, names in os.walk(out):
  for name in names:
   full=os.path.join(root,name)
   z.write(full, os.path.relpath(full, out))
print('학습된 어댑터를 다운로드합니다. 사용자 개인 저장 공간에 보관하세요.')
files.download(path)

## 그다음: 실제 마음결 웹사이트에 연결하기
학습된 LoRA 어댑터를 **지속적으로 실행되는 GPU 서버**에 배치해야 합니다. Render 무료 웹서비스에는 GPU가 없어 학습된 4B 모델을 직접 올릴 수 없습니다.

GPU 서버에 Python/CUDA 및 `training/requirements-gpu.txt`를 설치하고 어댑터 ZIP을 풀어 다음 명령으로 실행하세요.

```bash
export MODEL_API_KEY='충분히 긴 무작위 비밀키'
python training/serve_gpu.py --adapter /path/to/adapter --host 0.0.0.0 --port 8000
```

GPU 서버는 반드시 **HTTPS 역방향 프록시** 뒤에 두고 외부 직접 접근을 막으세요. 그런 다음 Render의 마음결 서비스 환경 변수에 아래 값을 입력하면 실제 모델 답변으로 전환됩니다. 비밀키를 GitHub에 올리지 마세요.

```env
MODEL_URL=https://GPU-SERVER.example/v1/chat/completions
MODEL_NAME=maumgyeol-qwen3-lora
MODEL_PROVIDER=openai
MODEL_API_KEY=GPU_서버에_설정한_키
```

이 노트북 실행만으로 Render 사이트가 자동 연결되지는 않습니다. 외부 서버 URL과 키가 필요합니다.